In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import sys
import os

PROJECT_ROOT = "/content/drive/MyDrive/isic-flagship-project"
os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

print(f"Working directory:", os.getcwd())

Working directory: /content/drive/MyDrive/isic-flagship-project


In [3]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/core/config.py
"""
Core configuration
"""

from pydantic_settings import BaseSettings
from functools import lru_cache

class Settings(BaseSettings):
    APP_NAME: str = "ISIC 2024 Skin Cancer Detection"
    API_VERSION: str = "v1"
    MODEL_VERSION: str = "2024-ensemble-v1"
    DEBUG: bool = True
    DATABASE_URL: str = "sqlite+aiosqlite:///./isic.db"
    NGROK_AUTHTOKEN: str = ""
    SECRET_KEY: str = "super-secret-key-change-in-production"

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"


@lru_cache()
def get_settings() -> Settings:
    return Settings()




Overwriting /content/drive/MyDrive/isic-flagship-project/src/core/config.py


In [4]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/schemas/__init__.py


Overwriting /content/drive/MyDrive/isic-flagship-project/src/schemas/__init__.py


In [5]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/schemas/prediction.py
"""
Pydantic schemas for API requests and responses
"""

from pydantic import BaseModel, Field
from datetime import datetime
from typing import Optional

class PredictionRequest(BaseModel):
    isic_id: Optional[str] = None

class PredictionResponse(BaseModel):
    isic_id: str
    probability: float = Field(..., ge=0.0, le=1.0)
    prediction: str
    model_version: str
    timestamp: datetime

class HealthResponse(BaseModel):
    status: str
    model_version: str
    api_version: str



Overwriting /content/drive/MyDrive/isic-flagship-project/src/schemas/prediction.py


In [6]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/services/__init__.py


Overwriting /content/drive/MyDrive/isic-flagship-project/src/services/__init__.py


In [7]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/services/prediction_service.py
"""
Business logic for predictions
"""

import torch
from torchvision import transforms
from PIL import Image
import io

from src.inference.inference_core import ISICInferenceEngine
from src.core.config import get_settings

class PredictionService:
    def __init__(self):
        self.settings = get_settings()
        self.engine = ISICInferenceEngine()
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225]),
        ])

    async def predict(self, image_bytes):
        """Process uploaded image and return prediction"""
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        image_tensor = self.transform(image).unsqueeze(0)

        prob = self.engine.predict_single_image(image_tensor)

        prediction = "Malignant" if prob > 0.5 else "Benign"

        return {
             "probability": float(prob),
             "prediction": prediction,
             "model_version": self.settings.MODEL_VERSION
        }





Overwriting /content/drive/MyDrive/isic-flagship-project/src/services/prediction_service.py


In [8]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/__init__.py


Overwriting /content/drive/MyDrive/isic-flagship-project/src/api/__init__.py


In [9]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/routes.py
"""
API Route definitions
"""

from fastapi import APIRouter, UploadFile, File, HTTPException
from src.schemas.prediction import PredictionResponse, HealthResponse
from src.services.prediction_service import PredictionService
from src.core.config import get_settings
from datetime import datetime

prediction_service = PredictionService()

health_router = APIRouter()
prediction_router = APIRouter()

@health_router.get("/health", response_model=HealthResponse)
async def health_check():
    settings = get_settings()
    return HealthResponse(
        status="healthy",
        model_version=settings.MODEL_VERSION,
        api_version=settings.API_VERSION
    )

@prediction_router.post("/predict", response_model=PredictionResponse)
async def predict_image(file: UploadFile = File(...)):
    if not file.content_type or not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="File must be an image")

    image_bytes = await file.read()
    result = await prediction_service.predict(image_bytes)

    return PredictionResponse(
        isic_id=file.filename or "uploaded_image",
        probability=result["probability"],
        prediction=result["prediction"],
        model_version=result["model_version"],
        timestamp=datetime.utcnow()
    )



Overwriting /content/drive/MyDrive/isic-flagship-project/src/api/routes.py


In [10]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/main.py
"""
Main FastAPI Application Entry Point
"""

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

from src.core.config import get_settings
from src.api.routes import health_router, prediction_router

settings = get_settings()

app = FastAPI(
    title=settings.APP_NAME,
    version=settings.API_VERSION,
    description="Production-ready ISIC 2024 1st/2nd Place Skin Cancer Detection API"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

app.include_router(health_router, prefix="/api/v1", tags=["health"])
app.include_router(prediction_router, prefix="/api/v1", tags=["prediction"])

@app.get("/")
async def root():
    return {
        "message": "ISIC 2024 Flagship API is running 🚀",
        "docs_url": "/docs"
    }



Overwriting /content/drive/MyDrive/isic-flagship-project/src/api/main.py


In [11]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/isic-flagship-project")
sys.path.insert(0, "/content/drive/MyDrive/isic-flagship-project/src")

from src.api.main import app
from src.core.config import get_settings


print(f"App Name: {get_settings().APP_NAME}")


Engine initialized on cuda
App Name: ISIC 2024 Skin Cancer Detection
